# 01 – Data audit

Checks the Roboflow pork-rasher dataset before any training:

1. class balance per split,
2. photos that appear in more than one split (leakage),
3. a reproducible manual review of images with no defect labels.

Run in Google Colab. Add your Roboflow **private** API key to Colab Secrets as `ROBOFLOW_API_KEY` (key icon on the left) first.

In [ ]:
# Get the code (public repo, no token needed) and install the requirements.
# Safe to re-run: pulls the latest code if the repo is already cloned.
import os, sys

REPO = "/content/MMA3001-Project-repository"
if not os.path.isdir(REPO):
    !git clone https://github.com/rhal0027/MMA3001-Project-repository.git {REPO}
else:
    !git -C {REPO} pull
%cd {REPO}
!pip install -q -r requirements.txt

# Make the package importable straight away (an editable pip install
# only takes effect in Colab after a runtime restart).
if f"{REPO}/src" not in sys.path:
    sys.path.insert(0, f"{REPO}/src")


In [ ]:
# Download the dataset into data/raw/ (git-ignored)
from google.colab import userdata
from rasher_inspection import dataset as ds
from rasher_inspection import visualise as vis

# Re-load in case an older version was imported earlier in this session (after a git pull)
import importlib
importlib.reload(ds); importlib.reload(vis)

root = ds.download_dataset(userdata.get("ROBOFLOW_API_KEY"), "data/raw/pork-rasher", version=4)
names = ds.load_class_names(root)
names

## 1. Class balance

In [ ]:
import pandas as pd

summary = ds.split_summary(root)
counts = pd.DataFrame({s: v["boxes"] for s, v in summary.items()})
info = pd.DataFrame({s: {k: v[k] for k in ("images", "sources", "empty_label", "missing_label")}
                     for s, v in summary.items()})
display(info)
display(counts)
counts.to_csv("results/class_counts.csv")
info.to_csv("results/split_info.csv")

## 2. Cross-split overlaps

Matched by Roboflow source id, so check each pair by eye: two different photos can share a file name.

In [ ]:
overlaps = ds.find_cross_split_overlaps(root)
print(len(overlaps), "source photos appear in more than one split")
vis.show_pairs(overlaps, class_names=names);

After looking at the pairs, list any that are **not** the same photo, then move the confirmed duplicates out of `train` (valid/test stay untouched). Run once with `dry_run=True` to check, then `False`.

In [ ]:
# Visual check (22 Oct 2026): all 13 file-name overlaps are DIFFERENT trays
# that happen to share a file name, so nothing is removed.
not_same_photo = ["1521_jpg", "2223_jpg", "2247_jpg", "3482_jpg", "6510_jpg",
                  "7277_jpg", "7506_jpg", "7508_jpg", "7510_jpg", "7555_jpg",
                  "7617_jpg", "8049_jpg", "9018_jpg"]
confirmed = [o for o in overlaps if o["source"] not in not_same_photo]

moved = ds.quarantine_overlaps(root, confirmed, "data/quarantine", from_split="train",
                               log_csv="results/leakage_removed.csv", dry_run=True)
print(len(confirmed), "confirmed duplicates;", len(moved), "train images would be moved")

## 2b. Content-based duplicate check

File names are not reliable here (the 13 matches above were all different trays), and a copy of the same tray under a *different* name would be missed. This compares the images themselves with a perceptual hash (dHash).

**First attempt (64-bit hash, threshold 6 bits):** 179 of 200 valid/test images were flagged, and the 20 closest pairs inspected were all *different* trays. The nearest-neighbour distances had no gap (median 3 bits): at 9×8 pixels the trays look almost identical. So the threshold must be **calibrated** rather than guessed.

**Calibration:** augmented copies in `train` share a source id, so they are *known* duplicates. Compare their distances with the distance from each image to its closest *different* tray. If the two distributions separate, the gap gives the threshold.

In [ ]:
import matplotlib.pyplot as plt

cal = {size: ds.hash_calibration(root, "train", size=size) for size in (8, 16)}   # ~1 min

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (size, c) in zip(axes, cal.items()):
    bins = range(0, max(c["same"] + c["different"]) + 2)
    ax.hist(c["different"], bins=bins, alpha=0.6, label="closest different tray")
    ax.hist(c["same"], bins=bins, alpha=0.6, label="augmented copy of same photo")
    ax.set_title(f"{size}x{size} dHash ({size*size} bits)")
    ax.set_xlabel("Hamming distance (bits)"); ax.set_ylabel("count"); ax.legend()
fig.tight_layout()
fig.savefig("results/dhash_calibration.png", dpi=150, bbox_inches="tight")
plt.show()

for size, c in cal.items():
    s, d = pd.Series(c["same"]), pd.Series(c["different"])
    print(f"{size}x{size}: same-photo max={s.max()}, 99th pct={s.quantile(.99):.0f} | "
          f"different-tray min={d.min()}, 1st pct={d.quantile(.01):.0f} | "
          f"different trays at or below same-photo max: {(d <= s.max()).sum()}")

**Result (22 Oct 2026):**

- **8×8 (64 bits):** the two distributions lie on top of each other. Useless for this dataset.
- **16×16 (256 bits):** known copies are mostly 15–40 bits apart, the closest *different* tray 28–70 bits. They overlap between about 28 and 40, so no threshold separates them cleanly. The long "same photo" tail up to 111 bits is pairs that share a file name but are different trays (the same name clash as in section 2), which also means source ids are not fully reliable labels.

Conclusion: dHash is **not** used as the leakage test. This is reported as a negative result, and the check moves to something that matches how the data was made (section 2c).

## 2c. Neighbouring video frames

The file names (`frame_with_red_4088_jpg…`) show the images are **frames from production-line video**. Consecutive frames show the *same tray* moved slightly along the line. So the real leakage risk is not an identical copy, but a test frame whose neighbours (frame ±1, ±2…) are in training: the model has effectively seen that tray.

First, how far is each valid/test frame from the nearest training frame number?

In [ ]:
import matplotlib.pyplot as plt

gaps = ds.frame_gaps(root)
g = pd.Series([r["gap"] for r in gaps if r["gap"] is not None])
print(len(gaps), "valid/test frames,", g.size, "with a training frame of the same video prefix")
print(g.describe())
for k in (0, 1, 2, 5, 10):
    print(f"  gap <= {k:2d}: {(g <= k).sum():3d} images ({(g <= k).mean():.0%})")

plt.hist(g.clip(upper=50), bins=range(0, 52))
plt.xlabel("frames to nearest training frame (50 = 50 or more)")
plt.ylabel("valid/test images")
plt.title("Distance from each evaluation frame to the nearest training frame")
plt.savefig("results/frame_gap_hist.png", dpi=150, bbox_inches="tight")
plt.show()

Frame numbers alone can mislead: the 13 file-name clashes (gap 0) were different trays, so some frame numbers restart between recordings. Look at the small-gap pairs to see whether neighbouring frames really show the same tray. (`d` in the titles is the frame gap.)

In [ ]:
MAX_GAP = 4
close = [r for r in gaps if r["gap"] is not None and 0 < r["gap"] <= MAX_GAP]
pairs = [{"query": r["query"], "match": r["nearest_image"], "query_split": r["query_split"],
          "match_split": "train", "distance": r["gap"]} for r in close]
print(len(pairs), "evaluation frames within", MAX_GAP, "frames of a training frame")
vis.show_matches(pairs, class_names=names, max_rows=20);   # page with pairs[20:40] etc.

Cross-check by content, independent of names: the pixel correlation (brightness-invariant, 96×72 greyscale) between each valid/test image and its most similar other image. 1.0 means identical apart from brightness/contrast.

In [ ]:
corr = ds.nearest_by_correlation(root)
r = pd.Series([m["correlation"] for m in corr])
print(r.describe())
plt.hist(r, bins=50)
plt.xlabel("correlation with most similar other image"); plt.ylabel("valid/test images")
plt.savefig("results/nearest_correlation_hist.png", dpi=150, bbox_inches="tight")
plt.show()
vis.show_matches(corr, class_names=names, max_rows=20);   # the 20 most similar pairs

In [ ]:
# Record what you saw. Row numbers refer to the figures above.
same_tray_frame_rows = []   # rows in the frame-gap figure that ARE the same tray
same_tray_corr_rows = []    # rows in the correlation figure that ARE the same tray

pd.DataFrame([{"query": p["query"].name, "match": p["match"].name, "frame_gap": p["distance"],
               "same_tray": i in same_tray_frame_rows} for i, p in enumerate(pairs)]
             ).to_csv("results/frame_neighbour_review.csv", index=False)
pd.DataFrame([{"query": m["query"].name, "match": m["match"].name, "correlation": round(m["correlation"], 4),
               "same_tray": i in same_tray_corr_rows} for i, m in enumerate(corr[:20])]
             ).to_csv("results/correlation_review.csv", index=False)
print(len(same_tray_frame_rows), "same-tray frame pairs;", len(same_tray_corr_rows), "same-tray correlation pairs")

## 3. Manual review of unlabelled images

Criteria (written before reviewing):

- **clean** – no visible defect of any of the five classes
- **missed_defect** – a defect is visible but has no box (note which class)
- **unclear** – glare, blur or framing makes it impossible to judge

**Test split (22 Oct 2026):** all 12 unlabelled test images reviewed: 9 clean, 3 missed defects. Many "clean" images are trays only partly in frame (entering or leaving the camera view), not complete defect-free trays.

Next: a sample of the 684 unlabelled **train** images.

In [ ]:
SPLIT, N, SEED = "train", 24, 1     # run once with ("test", 24, 0) too, to save the corrected test review

unl = ds.find_unlabelled(root, SPLIT)
print({k: len(v) for k, v in unl.items()})
sample = ds.sample_for_review(unl["empty_label"], N, seed=SEED)
sheet = ds.write_review_sheet(sample, f"results/unlabelled_review_{SPLIT}_seed{SEED}.csv")
vis.show_grid(sample, cols=6, title=f"Unlabelled {SPLIT} images (seed {SEED})");

In [ ]:
from rasher_inspection.stats import wilson_interval

# Verdicts keyed by the frame number in the file name (e.g. "4088" in frame_with_red_4088_jpg...).
# Anything not listed is recorded as "clean".
verdicts = {
    "test": {
        "4088": ("missed_defect", "packaging-error"),
        "7471": ("missed_defect", "unsealed"),
        "7325": ("missed_defect", "twisted-meat"),
    },
    "train": {
        # "1234": ("missed_defect", "loose-meat"),
        # "5678": ("unclear", "glare"),
    },
}

review = pd.read_csv(sheet)
number = review["image"].map(lambda n: str(ds.frame_number(n)[1]) if ds.frame_number(n) else "")
unknown = set(verdicts[SPLIT]) - set(number)
assert not unknown, f"verdict keys not in this sample: {unknown}"
v = number.map(lambda n: verdicts[SPLIT].get(n, ("clean", "")))
review["verdict"] = [x[0] for x in v]
review["notes"] = [x[1] for x in v]
review.to_csv(sheet, index=False)

missed = (review["verdict"] == "missed_defect").sum()
lo, hi = wilson_interval(missed, len(review))
print(review["verdict"].value_counts())
print(f"Missed-defect rate: {missed}/{len(review)} = {missed/len(review):.0%} "
      f"(95% CI {lo:.0%}-{hi:.0%})")
# Download results/*.csv and results/*.png from the Colab file browser and commit them.